In [106]:
pip install networkx
pip install python-louvain
pip install scikit-learn
pip install streamlit

SyntaxError: invalid syntax (267628983.py, line 1)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import re
import json
import sys
from pathlib import Path

In [ ]:
project_root = Path(r"C:\Users\yslog\OneDrive\Desktop\GenAI Project\jobprep-ai-backend")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from snowflake_connection import get_snowflake_connection
conn = get_snowflake_connection()

In [67]:
cur = conn.cursor()
cur.execute("""
SELECT table_catalog, table_schema, table_name
FROM JOBPREP_DB.INFORMATION_SCHEMA.TABLES
WHERE UPPER(table_name) = 'MART_QUESTION_BANK'
ORDER BY table_schema
""")
print(cur.fetchall())

[('JOBPREP_DB', 'DBT_JOBPREP_DBT_JOBPREP_MARTS', 'MART_QUESTION_BANK')]


In [ ]:
query_jobs = "SELECT * FROM JOBPREP_DB.DBT_JOBPREP_DBT_JOBPREP_MARTS.MART_QUESTION_BANK"
jobs_df = pd.read_sql(query_jobs, conn)
print(jobs_df.head())

In [ ]:
def clean_llm_output(text):
    # normalize to string
    if isinstance(text, list):
        text = "\n".join(map(str, text))
    elif isinstance(text, dict):
        text = json.dumps(text, ensure_ascii=False)
    elif text is None:
        return ""

    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()

In [ ]:
def llm(prompt, model="llama3.3-70b"):

    query = f"""
    SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
        '{model}',
        $$ {prompt} $$
    ) AS RESPONSE
    """

    result = pd.read_sql(query, conn)

    raw_text = result.iloc[0]["RESPONSE"]

    return clean_llm_output(raw_text)

In [ ]:
def embed(text):

    query = """
    SELECT SNOWFLAKE.CORTEX.EMBED_TEXT_768(
        'snowflake-arctic-embed-m-v1.5',
        %s
    ) AS EMB
    """

    cur = conn.cursor()

    try:
        cur.execute(query, (text,))
        result = cur.fetchone()[0]
    finally:
        cur.close()

    return result

In [ ]:
query_jobs = """
SELECT *
FROM JOBPREP_DB.DBT_JOBPREP_DBT_JOBPREP_MARTS.MART_QUESTION_BANK
"""

jobs_df = pd.read_sql(query_jobs, conn)

print(jobs_df.head())



In [ ]:
G = nx.Graph()

for _, row in jobs_df.iterrows():

    company = row["COMPANY_NAME"]
    role = row["ROLE_NAME"]
    question = row["INTERVIEW_QUESTION"]
    category = row["QUESTION_CATEGORY_ENHANCED"]

    # nodes
    G.add_node(company, type="company")
    G.add_node(role, type="role")
    G.add_node(question, type="question")
    G.add_node(category, type="category")

    # edges
    G.add_edge(company, role, relation="hires_for")
    G.add_edge(role, question, relation="asks")
    G.add_edge(question, category, relation="belongs_to")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
# detect communities
partition = community_louvain.best_partition(G)

print("Total communities detected:", len(set(partition.values())))

for cid, nodes in list(clusters.items())[:5]:
    print(f"\nCluster {cid}:")
    print(nodes[:10])

Total communities detected: 14


In [22]:
question_clusters = {}

for cid, nodes in clusters.items():

    questions = []

    for node in nodes:
        if G.nodes[node]["type"] == "question":
            questions.append(node)

    question_clusters[cid] = questions

In [ ]:
def summarize_cluster(questions):

    text = "\n".join(questions[:10])

    prompt = f"""
These interview questions belong to the same topic:

{text}

Identify:
Topic:
Role/Domain:
Skills:
Summary:
"""

    return llm(prompt)

In [ ]:
cluster_summaries = {}

for cid, qs in question_clusters.items():

    if len(qs) > 0:
        summary = summarize_cluster(qs)
        cluster_summaries[cid] = summary

summary_df = pd.DataFrame(
    list(cluster_summaries.items()),
    columns=["cluster_id", "summary"]
)

print(summary_df)

C:\Users\yslog\AppData\Local\Temp\ipykernel_5040\1917212139.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [47]:
summary_df["embedding"] = summary_df["summary"].apply(embed)

In [ ]:
def build_user_query(company, role, job_description):
    return f"""
Company: {company}
Role: {role}

Job Description:
{job_description}

Generate a mock interview aligned with this company, role, and job description.
"""

In [ ]:
def retrieve_top_clusters(company, role, job_description, summary_df, top_k=3):
    user_query = build_user_query(company, role, job_description)
    query_embedding = embed(user_query)

    cluster_embeddings = np.vstack(summary_df["embedding"].values)
    scores = cosine_similarity([query_embedding], cluster_embeddings)[0]

    ranked = summary_df.copy()
    ranked["score"] = scores

    return ranked.sort_values("score", ascending=False).head(top_k)

In [ ]:
cluster_embeddings = np.vstack(summary_df["embedding"].values)

scores = cosine_similarity([query_embedding], cluster_embeddings)

summary_df["score"] = scores[0]

In [94]:
def generate_mock_interview(company, role, job_description, selected_questions):
    text = "\n".join(selected_questions)

    prompt = f"""
You are an expert technical interviewer.

Generate a realistic mock interview based on:

Company: {company}
Role: {role}
Job Description:
{job_description}

Use these example historical interview questions as grounding:
{text}

Return:
- 3 coding questions
- 1 system design question
- 1 behavioral question

Requirements:
- align with the company and role
- reflect the job description
- keep the interview realistic
- avoid copying the example questions exactly unless necessary
- format clearly with section headers
"""

    return llm(prompt, model="deepseek-r1")

In [99]:
def collect_relevant_questions(top_clusters, jobs_df, cluster_questions, company, role, max_questions=12):
    selected_questions = []

    exact_df = jobs_df[
        (jobs_df["COMPANY_NAME"].fillna("").str.lower() == company.lower()) &
        (jobs_df["ROLE_NAME"].fillna("").str.lower() == role.lower())
    ]
    exact_questions = set(exact_df["INTERVIEW_QUESTION"].dropna().tolist())

    role_df = jobs_df[
        jobs_df["ROLE_NAME"].fillna("").str.lower() == role.lower()
    ]
    role_questions = set(role_df["INTERVIEW_QUESTION"].dropna().tolist())

    for cid in top_clusters["cluster_id"]:
        for q in cluster_questions.get(cid, []):
            if q in exact_questions and q not in selected_questions:
                selected_questions.append(q)

    if len(selected_questions) < max_questions:
        for cid in top_clusters["cluster_id"]:
            for q in cluster_questions.get(cid, []):
                if q in role_questions and q not in selected_questions:
                    selected_questions.append(q)

    if len(selected_questions) < max_questions:
        for cid in top_clusters["cluster_id"]:
            for q in cluster_questions.get(cid, []):
                if q not in selected_questions:
                    selected_questions.append(q)

    return selected_questions[:max_questions]

In [100]:
def run_graphrag_interview(company, role, job_description, summary_df, jobs_df, cluster_questions, top_k=3):
    top_clusters = retrieve_top_clusters(
        company=company,
        role=role,
        job_description=job_description,
        summary_df=summary_df,
        top_k=top_k
    )

    selected_questions = collect_relevant_questions(
        top_clusters=top_clusters,
        jobs_df=jobs_df,
        cluster_questions=cluster_questions,
        company=company,
        role=role,
        max_questions=12
    )

    interview = generate_mock_interview(
        company=company,
        role=role,
        job_description=job_description,
        selected_questions=selected_questions
    )

    return top_clusters, selected_questions, interview

In [101]:
top_clusters, selected_questions, interview = run_graphrag_interview(
    company="Google",
    role="Software Engineer",
    job_description="""
Strong data structures and algorithms, scalable backend systems,
distributed systems knowledge, debugging, and teamwork.
""",
    summary_df=summary_df,
    jobs_df=jobs_df,
    cluster_questions=question_clusters,
    top_k=3
)

print(top_clusters[["cluster_id", "summary", "score"]])

print("\nSelected questions:\n")
for q in selected_questions[:8]:
    print("-", q)

print("\nFinal interview:\n")
print(interview)

C:\Users\yslog\AppData\Local\Temp\ipykernel_5040\2764797227.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


    cluster_id                                            summary     score
13           7  "The main interview topic appears to be a tech...  0.694900
12           6  "The main interview topic is Product Managemen...  0.684335
0            0  "The main interview topic appears to be a tech...  0.682468

Selected questions:

- Count Operations to Obtain Zero
- Contains Duplicate II
- Contains Duplicate
- Count All Valid Pickup and Delivery Options
- Accounts Merge
- Count and Say
- Design HashSet
- Insert Delete GetRandom O(1) - Duplicates allowed

Final interview:

"\n\n### Coding Questions\n\n**1. Design a Frequency-Based Stack**  \nDesign a data structure that simulates a stack with the following operations:  \n- `push(val)`: Pushes `val` onto the stack.  \n- `pop()`: Removes and returns the *most frequent element*. If multiple elements have the same highest frequency, the element closest to the top of the stack is returned.  \n\nBoth operations must run in **O(1)** average time.  \n